# Folium Mapping Tutorial
This notebook walks through the workflow for building an interactive compaction map from CKAN-discovered Upstream station resources. The Folium markers point to a reusable popup page that fetches each station's Upstream measurement JSON directly when the popup opens.

CKAN plumbing lives in `utils.py` so the notebook can focus on the Folium, popup, and data-loading concepts students need to see directly.


In [25]:
import json
import os
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
import numpy as np
import geopandas as gpd

import plotly.express as px
from plotly import graph_objects as go
from plotly.subplots import make_subplots

import folium
import branca
from branca.element import MacroElement
from jinja2 import Template
import requests

import utils as tutorial_utils


## Load Data
Start by importing the libraries used in the workflow. `pandas` and `geopandas` handle tabular and spatial data, `plotly` creates the popup charts, and `folium` builds the interactive web map.


### Read County Boundaries
Load the Texas county GeoJSON into a GeoDataFrame so selected counties can be drawn as overlays on the Folium map.


#### Find Data From Ckan

In [26]:
txgeojson='https://ckan.tacc.utexas.edu/dataset/cd3deceb-7102-44b1-a83b-35da7c8f6855/resource/204f8874-0db4-4e81-95e3-e695f4056bdc/download/texas_county_boundaries_detailed.geojson'
tx_gdf_county = gpd.read_file(txgeojson)

### Check the Spatial Layer
Filter to one county as a quick validation step. This is useful for confirming the boundary file loaded correctly and that the county names match what you expect to use later.


In [27]:
tx_gdf_county[tx_gdf_county['CNTY_NM']=='Fort Bend']

,OBJECTID,CMPTRL_CNTY_NBR,DPS_CNTY_NBR,FIPS_ST_CNTY_CD,TXDOT_CNTY_NBR,TXDOT_DIST_NBR,CNTY_NM,CNTY_NBR,DIST_NBR,GID,geometry
8,9,79,79,48157,80,12,Fort Bend,80,12,43,"POLYGON ((-95.78398 29.765, -95.76693 29.75549..."


### Discover Upstream Station Resources from CKAN
Query CKAN for the Houston-area extensometer campaign packages, then build a station table from the package metadata and each package's Upstream measurement resource URL.


In [28]:
CKAN_URL = 'https://ckan.tacc.utexas.edu'
CAMPAIGN_TAG = 'houston-area-extensometer-compaction-campaign'

station_packages = tutorial_utils.discover_station_packages(
    ckan_url=CKAN_URL,
    campaign_tag=CAMPAIGN_TAG,
)
sites_df = tutorial_utils.build_sites_dataframe(station_packages)


### Inspect the Site Table
Preview the CKAN-derived station table and confirm each site has a browser-accessible Upstream measurement URL.


In [29]:
sites_df[['station_id', 'sensor_id', 'GENERAL_NM', 'STATION_NM', 'measurement_url']].head()


,station_id,sensor_id,GENERAL_NM,STATION_NM,measurement_url
0,4,16,Texas City,KH-64-33-920 (Texas City Extensometer),https://upstreamapi.pods.portals.tapis.io/api/...
1,5,17,Lake Houston,LJ-65-07-909 (Lake Houston Extensometer),https://upstreamapi.pods.portals.tapis.io/api/...
2,6,18,Addicks,LJ-65-12-726 (Addicks Extensometer),https://upstreamapi.pods.portals.tapis.io/api/...
3,7,19,Northeast,LJ-65-14-746 (Northeast Extensometer),https://upstreamapi.pods.portals.tapis.io/api/...
4,8,20,Baytown Shallow,LJ-65-16-930 (Baytown C-1 Extensometer),https://upstreamapi.pods.portals.tapis.io/api/...


### Load Compaction Measurements for Notebook Preview
Fetch each Upstream measurement endpoint once so the notebook can preview the same data that the popup HTML will fetch in the browser.


### Fetch Upstream Measurement Pages
This function handles pagination for a single Upstream measurement endpoint. Keeping it separate makes the network step explicit: one station can have many pages of JSON records, and the rest of the notebook should work with a plain Python list.


In [30]:
def fetch_measurement_items(measurement_url, *, page_size=1000):
    page = 1
    items = []
    while True:
        response = requests.get(measurement_url, params={'limit': page_size, 'page': page}, timeout=60)
        response.raise_for_status()
        payload = response.json()
        page_items = payload.get('items', [])
        items.extend(page_items)

        total_pages = payload.get('pages') or page
        if page >= total_pages or len(page_items) < page_size:
            break
        page += 1
    return items


### Convert One Site to the Tutorial Data Columns
This function turns the raw Upstream JSON for one station into the columns used throughout the map: measurement date, cumulative compaction, site name, and a simple data version. That keeps the downstream plotting and mapping cells independent of the API response shape.


In [31]:
def load_measurements_from_site(site_row):
    items = fetch_measurement_items(site_row['measurement_url'])
    if not items:
        return pd.DataFrame(columns=['DATE', 'CUMULATIVE_COMPACTION', 'site', 'data_version'])

    measurements = pd.DataFrame(items)
    measurements['DATE'] = pd.to_datetime(measurements['collectiontime'])
    measurements['CUMULATIVE_COMPACTION'] = pd.to_numeric(measurements['value'], errors='coerce')
    measurements['site'] = site_row['name_condensed']
    measurements['data_version'] = measurements['DATE'].dt.year.max()
    return measurements[['DATE', 'CUMULATIVE_COMPACTION', 'site', 'data_version']]


### Load Measurements for All Sites
Now that one station can be normalized, loop over the CKAN-discovered station table and combine every station into one dataframe for preview charts and map popups.


In [32]:
measurement_frames = [load_measurements_from_site(row) for _, row in sites_df.iterrows()]
df = pd.concat(measurement_frames, ignore_index=True).sort_values(['site', 'DATE']).reset_index(drop=True)


### Clean the Date Column
The Upstream JSON already provides ISO-like timestamps. Convert them to datetimes and keep the table sorted for plotting.


In [33]:
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.sort_values(['site', 'DATE']).reset_index(drop=True)
df.head()


,DATE,CUMULATIVE_COMPACTION,site,data_version
0,1974-07-11,0.000,Addicks,2023
1,1974-07-24,0.004,Addicks,2023
2,1974-08-30,0.009,Addicks,2023
3,1974-09-30,0.010,Addicks,2023
4,1974-10-30,0.020,Addicks,2023


##  CKAN Registration Checkpoint

Before moving into popup generation and map assembly, this is still a good point to review the CKAN station packages and the Upstream measurement resources discovered from them. The popup HTML generated below will use those measurement URLs directly rather than embedding a static copy of the data.


### Recommended Registration Targets

Treat the Upstream measurement endpoints as the source data product, then create or update a CKAN dataset for this notebook run's generated visualization artifacts. The output dataset holds `popup.html` and `index.html`, while the marker popups still fetch the source measurements directly from Upstream.


## Build and Preview a Site Chart
Before automating the map popups, isolate one site and make a simple Plotly figure. This lets you confirm the compaction time series looks right for a single location first.


### Subset a Single Site
Start with the `TexasCity` site so you can test the plotting workflow on one location before looping through the entire dataset.


In [34]:
tc = df[df.site=='TexasCity']
tc.head(2)

,DATE,CUMULATIVE_COMPACTION,site,data_version
7546,1973-07-13,0.000,TexasCity,2023
7547,1973-08-13,0.007,TexasCity,2023


### Create a Quick Plotly Figure
Build a simple scatter plot of cumulative compaction over time. This is a first visual check before defining a reusable chart function.


In [35]:
fig = px.scatter(tc, x="DATE", y="CUMULATIVE_COMPACTION")
fig

### Define the Popup Plot Function
This function converts one station's cleaned time series into the Plotly chart used in the popup. It stays in the notebook because it is part of the learning goal: students can see exactly how the dataframe columns become an interactive chart.


In [36]:
def make_popup_figure(site, sites_df, compaction_df):
    sitename = sites_df.loc[sites_df['name_condensed'] == site, 'GENERAL_NM'].item()
    plot_data = compaction_df[compaction_df.site == site]
    title = 'Cumulative Compaction at ' + sitename + ' extensometer'
    fig = px.line(plot_data, x="DATE", y="CUMULATIVE_COMPACTION",
                 title=title,
                 labels={
                     "DATE": 'Measurement date',
                     "CUMULATIVE_COMPACTION": 'Cumulative Compaction (feet)'
                 }
                )
    fig.update_layout(
        title_font_weight='bold'
    )
    return fig

### Test the Reusable Figure Function
Call the helper once for `TexasCity` to verify it produces the expected popup chart before running it in a loop.


In [37]:
make_popup_figure('TexasCity',sites_df, df)

## Build a Reusable Popup Page
Each map marker will open the same small HTML page. The marker passes its Upstream measurement URL in the iframe query string, and the browser fetches the JSON directly when the popup opens.


### Prepare Output Storage
Create a folder for the popup template and final Folium map if it does not already exist.


In [38]:
output_dir = Path('folium_html')
output_dir.mkdir(exist_ok=True)

popup_template_path = output_dir / 'popup.html'
popup_template_path


PosixPath('folium_html/popup.html')

### Write the Browser-Fetched Popup Template
The template contains only rendering code. It reads the `url`, `site`, and `title` query parameters, fetches the Upstream JSON endpoint, and renders the Plotly chart in the browser.


In [39]:
popup_template = '''<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Compaction popup</title>
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
  <style>
    html, body {
      height: 100%;
      margin: 0;
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      color: #24313d;
      background: #ffffff;
    }
    #status {
      box-sizing: border-box;
      min-height: 28px;
      padding: 8px 10px 0;
      font-size: 12px;
      color: #53616f;
    }
    #chart {
      width: 100%;
      height: calc(100% - 28px);
      min-height: 310px;
    }
    .error {
      color: #9b1c1c;
    }
  </style>
</head>
<body>
  <div id="status">Loading measurements...</div>
  <div id="chart"></div>
  <script>
    const params = new URLSearchParams(window.location.search);
    const measurementUrl = params.get("url");
    const site = params.get("site") || "site";
    const title = params.get("title") || `Cumulative Compaction at ${site}`;
    const statusEl = document.getElementById("status");

    function fail(message) {
      statusEl.textContent = message;
      statusEl.className = "error";
    }

    function validMeasurementUrl(url) {
      return url && url.startsWith("https://upstreamapi.pods.portals.tapis.io/api/v1/");
    }

    async function fetchMeasurementItems(url) {
      const allItems = [];
      let page = 1;
      let totalPages = 1;

      while (page <= totalPages) {
        const pageUrl = new URL(url);
        pageUrl.searchParams.set("limit", "1000");
        pageUrl.searchParams.set("page", String(page));

        const response = await fetch(pageUrl.toString());
        if (!response.ok) {
          throw new Error(`HTTP ${response.status}`);
        }

        const payload = await response.json();
        const pageItems = Array.isArray(payload.items) ? payload.items : [];
        allItems.push(...pageItems);
        totalPages = Number(payload.pages || page);

        if (!pageItems.length) {
          break;
        }
        page += 1;
      }

      return allItems;
    }

    async function render() {
      if (!validMeasurementUrl(measurementUrl)) {
        fail("Missing or unsupported Upstream measurement URL.");
        return;
      }

      try {
        const items = await fetchMeasurementItems(measurementUrl);
        if (!items.length) {
          fail("No measurements returned from Upstream.");
          return;
        }

        items.sort((a, b) => new Date(a.collectiontime) - new Date(b.collectiontime));
        const x = items.map((item) => item.collectiontime);
        const y = items.map((item) => Number(item.value));
        const latest = new Date(x[x.length - 1]);

        statusEl.textContent = `${items.length} measurements fetched from Upstream. Latest: ${latest.toISOString().slice(0, 10)}`;
        statusEl.className = "";

        Plotly.newPlot("chart", [{
          x,
          y,
          type: "scatter",
          mode: "lines+markers",
          line: { color: "#2563a8", width: 2 },
          marker: { color: "#2563a8", size: 5 },
          hovertemplate: "%{x|%Y-%m-%d}<br>%{y:.3f} ft<extra></extra>"
        }], {
          title: { text: title, font: { size: 15 } },
          margin: { l: 56, r: 18, t: 54, b: 46 },
          xaxis: { title: "Measurement date" },
          yaxis: { title: "Cumulative Compaction (feet)" },
          template: "plotly_white",
          hovermode: "x unified"
        }, {
          responsive: true,
          displaylogo: false
        });
      } catch (error) {
        fail(`Could not load Upstream measurements: ${error.message}`);
      }
    }

    render();
  </script>
</body>
</html>
'''

popup_template_path.write_text(popup_template, encoding='utf-8')


3759

### Build a Popup Template Upload Record
Create a one-row table for the reusable popup file that will be uploaded to CKAN.


In [40]:
popup_template_df = pd.DataFrame([{
    'html_file': popup_template_path.name,
    'html_path': popup_template_path.as_posix(),
    'size_kb': round(popup_template_path.stat().st_size / 1024, 1),
}])
popup_template_df


,html_file,html_path,size_kb
0,popup.html,folium_html/popup.html,3.7


### Build Popup URLs for Map Markers
This function creates the query string passed into the reusable popup page. Each marker reuses the same hosted HTML template, but sends its own Upstream measurement URL, site id, and chart title.


In [41]:
def build_popup_url(template_url, site_row):
    params = {
        'url': site_row['measurement_url'],
        'site': site_row['name_condensed'],
        'title': f"Cumulative Compaction at {site_row['GENERAL_NM']} extensometer",
    }
    return f"{template_url}?{urlencode(params)}"


### Preview Popup URL Inputs
Before the template is uploaded, confirm that each site has the two fields the popup URL function needs: a compact site name and an Upstream measurement endpoint.


In [42]:
sites_df[['name_condensed', 'measurement_url']].head()


,name_condensed,measurement_url
0,TexasCity,https://upstreamapi.pods.portals.tapis.io/api/...
1,LakeHouston,https://upstreamapi.pods.portals.tapis.io/api/...
2,Addicks,https://upstreamapi.pods.portals.tapis.io/api/...
3,Northeast,https://upstreamapi.pods.portals.tapis.io/api/...
4,BaytownShallow,https://upstreamapi.pods.portals.tapis.io/api/...


### Verify the Popup Template
Preview the local template file size. The final popup URLs are created after the template is uploaded to CKAN.


In [43]:
popup_template_df


,html_file,html_path,size_kb
0,popup.html,folium_html/popup.html,3.7


## Publish Popup HTML to CKAN
Use the notebook's local CKAN helpers to authenticate with TACC, upload or update the reusable popup template, and build one browser-fetched popup URL per site.


The CKAN upload details are handled by `utils.py`. The next cell keeps all CKAN package metadata in one place, then the publish cell creates the dataset and uploads the generated HTML resources.


### Configure CKAN Dataset Metadata
This cell defines the CKAN package that will be created or updated for the generated visualization outputs. The required package fields are explicit here, while spatial and temporal coverage are derived from the county layer and loaded measurement dataframe so the metadata reflects the map students just built.


In [44]:
CKAN_URL = 'https://ckan.tacc.utexas.edu'
TAPIS_TOKEN_URL = 'https://portals.tapis.io/v3/oauth2/tokens'
MAX_UPLOAD_MB_WARNING = 5

CKAN_OUTPUT_OWNER_ORG = '2f68b69f-95b8-468c-b0c0-39d916f26c61'
CKAN_OUTPUT_DATASET_NAME = f'{CAMPAIGN_TAG}-folium-map'
CKAN_OUTPUT_DATASET_TITLE = 'Houston Area Extensometer Compaction Folium Map'
CKAN_OUTPUT_DATASET_NOTES = (
    'Interactive Folium map for the Houston-area extensometer compaction campaign. '
    'The dataset contains generated HTML visualization resources: a reusable Plotly popup shell and a Folium overview map. '
    'Map markers fetch cumulative compaction measurements directly from Upstream measurement JSON endpoints discovered from CKAN station packages.'
)
CKAN_OUTPUT_DATASET_URL = f'{CKAN_URL}/dataset/{CKAN_OUTPUT_DATASET_NAME}'
CKAN_OUTPUT_DATASET_AUTHOR = '<Your Name>'
CKAN_OUTPUT_DATASET_AUTHOR_EMAIL = '<Fake Email>'
CKAN_OUTPUT_DATASET_MAINTAINER = 'William Mobley'
CKAN_OUTPUT_DATASET_MAINTAINER_EMAIL = 'fake_email@tacc.utexas.edu'
CKAN_OUTPUT_DATASET_LICENSE_ID = 'cc-by'
CKAN_OUTPUT_DATASET_VERSION = '1.0'
CKAN_OUTPUT_DATASET_TYPE = 'dataset'
CKAN_OUTPUT_DATASET_ISOPEN = True
CKAN_OUTPUT_DATASET_PRIVATE = False
CKAN_OUTPUT_DATASET_TAGS = [
    'folium',
    'extensometer',
    'compaction',
    'subsidence',
    'upstream',
    'houston',
    'tutorial',
]

metadata_counties = ['Harris', 'Fort Bend', 'Galveston']
minx, miny, maxx, maxy = tx_gdf_county[tx_gdf_county['CNTY_NM'].isin(metadata_counties)].total_bounds
CKAN_OUTPUT_DATASET_SPATIAL = json.dumps({
    'type': 'Polygon',
    'coordinates': [[
        [float(minx), float(miny)],
        [float(minx), float(maxy)],
        [float(maxx), float(maxy)],
        [float(maxx), float(miny)],
        [float(minx), float(miny)],
    ]],
}, separators=(',', ':'))

CKAN_OUTPUT_TEMPORAL_COVERAGE_START = df['DATE'].min().date().isoformat()
CKAN_OUTPUT_TEMPORAL_COVERAGE_END = df['DATE'].max().date().isoformat()
CKAN_OUTPUT_DATASET_EXTRAS = [
    {'key': 'source_campaign_tag', 'value': CAMPAIGN_TAG},
    {'key': 'source_station_package_count', 'value': str(len(station_packages))},
    {'key': 'mapped_station_count', 'value': str(len(sites_df))},
    {'key': 'mapped_counties', 'value': ', '.join(metadata_counties)},
    {'key': 'measurement_endpoint_count', 'value': str(sites_df['measurement_url'].nunique())},
    {'key': 'generated_resources', 'value': 'popup.html,index.html'},
]

{
    'name': CKAN_OUTPUT_DATASET_NAME,
    'title': CKAN_OUTPUT_DATASET_TITLE,
    'spatial': CKAN_OUTPUT_DATASET_SPATIAL,
    'temporal_coverage_start': CKAN_OUTPUT_TEMPORAL_COVERAGE_START,
    'temporal_coverage_end': CKAN_OUTPUT_TEMPORAL_COVERAGE_END,
    'tags': CKAN_OUTPUT_DATASET_TAGS,
}


{'name': 'houston-area-extensometer-compaction-campaign-folium-map',
 'title': 'Houston Area Extensometer Compaction Folium Map',
 'spatial': '{"type":"Polygon","coordinates":[[[-96.0891771827673,29.0808504196017],[-96.0891771827673,30.1708652741932],[-94.3707067324284,30.1708652741932],[-94.3707067324284,29.0808504196017],[-96.0891771827673,29.0808504196017]]]}',
 'temporal_coverage_start': '1973-07-13',
 'temporal_coverage_end': '2023-12-29',
 'tags': ['folium',
  'extensometer',
  'compaction',
  'subsidence',
  'upstream',
  'houston',
  'tutorial']}

In [45]:
upload_size_df = popup_template_df.assign(
    size_mb=popup_template_df['html_path'].map(lambda p: round(tutorial_utils.file_size_mb(p), 3))
)
upload_size_df

ckan_client = tutorial_utils.build_ckan_client(
    ckan_url=CKAN_URL,
    tapis_token_url=TAPIS_TOKEN_URL,
)
html_dataset = tutorial_utils.create_output_dataset(
    ckan_client,
    name=CKAN_OUTPUT_DATASET_NAME,
    title=CKAN_OUTPUT_DATASET_TITLE,
    notes=CKAN_OUTPUT_DATASET_NOTES,
    owner_org=CKAN_OUTPUT_OWNER_ORG,
    tags=CKAN_OUTPUT_DATASET_TAGS,
    private=CKAN_OUTPUT_DATASET_PRIVATE,
    author=CKAN_OUTPUT_DATASET_AUTHOR,
    author_email=CKAN_OUTPUT_DATASET_AUTHOR_EMAIL,
    maintainer=CKAN_OUTPUT_DATASET_MAINTAINER,
    maintainer_email=CKAN_OUTPUT_DATASET_MAINTAINER_EMAIL,
    license_id=CKAN_OUTPUT_DATASET_LICENSE_ID,
    url=CKAN_OUTPUT_DATASET_URL,
    version=CKAN_OUTPUT_DATASET_VERSION,
    dataset_type=CKAN_OUTPUT_DATASET_TYPE,
    isopen=CKAN_OUTPUT_DATASET_ISOPEN,
    spatial=CKAN_OUTPUT_DATASET_SPATIAL,
    temporal_coverage_start=CKAN_OUTPUT_TEMPORAL_COVERAGE_START,
    temporal_coverage_end=CKAN_OUTPUT_TEMPORAL_COVERAGE_END,
    extras=CKAN_OUTPUT_DATASET_EXTRAS,
)

popup_template_resource = tutorial_utils.upsert_resource_by_name(
    ckan_client,
    html_dataset,
    popup_template_path,
    name=popup_template_path.name,
    description='Reusable Plotly popup shell that fetches Upstream measurement JSON in the browser.',
    format_name='HTML',
    max_upload_mb_warning=MAX_UPLOAD_MB_WARNING,
)
popup_template_url = tutorial_utils.resource_download_url(CKAN_URL, html_dataset, popup_template_resource)

sites_df = sites_df.drop(columns=['popup_resource_id', 'popup_url'], errors='ignore').copy()
sites_df['popup_resource_id'] = popup_template_resource['id']
sites_df['popup_url'] = sites_df.apply(lambda row: build_popup_url(popup_template_url, row), axis=1)
{
    'new_dataset_name': html_dataset['name'],
    'new_dataset_url': f"{CKAN_URL}/dataset/{html_dataset['name']}",
    'popup_template_url': popup_template_url,
    'preview': sites_df[['name_condensed', 'measurement_url', 'popup_url']].head(2),
}


{'new_dataset_name': 'houston-area-extensometer-compaction-campaign-folium-map',
 'new_dataset_url': 'https://ckan.tacc.utexas.edu/dataset/houston-area-extensometer-compaction-campaign-folium-map',
 'popup_template_url': 'https://ckan.tacc.utexas.edu/dataset/houston-area-extensometer-compaction-campaign-folium-map/resource/8d5da8ee-51d8-43d4-b9f0-c00df7b09da0/download/popup.html',
 'preview':   name_condensed                                    measurement_url  \
 0      TexasCity  https://upstreamapi.pods.portals.tapis.io/api/...   
 1    LakeHouston  https://upstreamapi.pods.portals.tapis.io/api/...   
 
                                            popup_url  
 0  https://ckan.tacc.utexas.edu/dataset/houston-a...  
 1  https://ckan.tacc.utexas.edu/dataset/houston-a...  }

## Create the Folium Map
With a CKAN-hosted popup template available, assemble the interactive map with base layers, the CKAN-hosted OPERA subsidence COG overlay, extensometer markers, and county outlines. Then save the finished map HTML locally for upload.


### Define the OPERA COG Overlay
Folium does not load Cloud Optimized GeoTIFFs by itself. This small `MacroElement` injects the browser-side JavaScript libraries that read the COG, colorize finite raster values, and add the result as a Leaflet layer.


In [ ]:
class GeoTiffOverlay(MacroElement):
    def __init__(self, cog_url, layer_group, *, opacity=0.65, legend_title='OPERA raster value', legend_units='mm/year'):
        super().__init__()
        self._name = 'GeoTiffOverlay'
        self.cog_url = cog_url
        self.layer_group = layer_group
        self.opacity = opacity
        self.legend_title = legend_title
        self.legend_units = legend_units
        self._template = Template("""
            {% macro header(this, kwargs) %}
            <script src="https://unpkg.com/georaster"></script>
            <script src="https://unpkg.com/georaster-layer-for-leaflet"></script>
            <script src="https://unpkg.com/chroma-js@2.4.2/chroma.min.js"></script>
            {% endmacro %}
            {% macro script(this, kwargs) %}
            (function() {
                const cogUrl = {{ this.cog_url|tojson }};
                const layerGroup = {{ this.layer_group.get_name() }};
                const opacity = {{ this.opacity }};
                const legendTitle = {{ this.legend_title|tojson }};
                const legendUnits = {{ this.legend_units|tojson }};
                const operaColorScale = chroma.scale(['#440154', '#21918c', '#fde725']);

                function finiteRasterRange(georaster) {
                    const raster = georaster.values[0];
                    const noDataValue = georaster.noDataValue;
                    let min = Infinity;
                    let max = -Infinity;
                    for (const row of raster) {
                        for (const value of row) {
                            if (Number.isFinite(value) && value !== noDataValue) {
                                min = Math.min(min, value);
                                max = Math.max(max, value);
                            }
                        }
                    }
                    if (!Number.isFinite(min) || !Number.isFinite(max)) {
                        return null;
                    }
                    if (min === max) {
                        min -= 1;
                        max += 1;
                    }
                    return { min, max };
                }

                function formatValue(value) {
                    if (Math.abs(value) >= 100 || Math.abs(value) < 0.01) {
                        return value.toExponential(2) + ' ' + legendUnits;
                    }
                    return value.toFixed(2) + ' ' + legendUnits;
                }

                function updateOperaLegend(range, attempt = 0) {
                    const legend = document.querySelector('.combined-map-legend');
                    if (!legend) {
                        if (attempt < 20) {
                            setTimeout(function() { updateOperaLegend(range, attempt + 1); }, 100);
                        }
                        return;
                    }
                    const title = legend.querySelector('.opera-title');
                    const minLabel = legend.querySelector('.opera-min');
                    const maxLabel = legend.querySelector('.opera-max');
                    if (title) {
                        title.textContent = legendTitle + ' (' + legendUnits + ')';
                    }
                    if (minLabel) {
                        minLabel.textContent = formatValue(range.min);
                    }
                    if (maxLabel) {
                        maxLabel.textContent = formatValue(range.max);
                    }
                }

                fetch(cogUrl)
                    .then(function(response) {
                        if (!response.ok) {
                            throw new Error('HTTP ' + response.status);
                        }
                        return response.arrayBuffer();
                    })
                    .then(parseGeoraster)
                    .then(function(georaster) {
                        const range = finiteRasterRange(georaster);
                        if (!range) {
                            console.warn('OPERA COG layer has no finite raster values.');
                            return;
                        }
                        const noDataValue = georaster.noDataValue;
                        const layer = new GeoRasterLayer({
                            georaster: georaster,
                            opacity: opacity,
                            resolution: 128,
                            pixelValuesToColorFn: function(values) {
                                const value = values[0];
                                if (!Number.isFinite(value) || value === noDataValue) {
                                    return null;
                                }
                                const normalized = (value - range.min) / (range.max - range.min);
                                return operaColorScale(Math.max(0, Math.min(1, normalized))).hex();
                            }
                        });
                        layer.addTo(layerGroup);
                        updateOperaLegend(range);
                    })
                    .catch(function(error) {
                        console.error('Could not load OPERA COG layer:', error);
                    });
            })();
            {% endmacro %}
        """)


### Define the Year-End Compaction Slider
This control adds a time slider for the point layer. For each year, the map colors every extensometer by the cumulative compaction value available on or before December 31 of that year, and the legend updates to show the selected year.


In [ ]:
class YearEndCompactionControl(MacroElement):
    def __init__(self, years, marker_refs, *, vmin, vmax, legend_title='Year-end cumulative compaction', legend_units='ft'):
        super().__init__()
        self._name = 'YearEndCompactionControl'
        self.years = years
        self.marker_refs = marker_refs
        self.vmin = vmin
        self.vmax = vmax
        self.legend_title = legend_title
        self.legend_units = legend_units
        self._template = Template("""
            {% macro header(this, kwargs) %}
            <script src="https://unpkg.com/chroma-js@2.4.2/chroma.min.js"></script>
            <style>
              .combined-map-legend {
                padding: 10px 11px;
                background: rgba(255, 255, 255, 0.96);
                border: 1px solid rgba(0, 0, 0, 0.22);
                border-radius: 4px;
                box-shadow: 0 1px 5px rgba(0, 0, 0, 0.18);
                color: #263238;
                font: 12px/1.35 -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
                min-width: 285px;
              }
              .combined-map-legend .legend-heading {
                font-weight: 700;
                margin-bottom: 7px;
              }
              .combined-map-legend .legend-section + .legend-section {
                margin-top: 10px;
                padding-top: 8px;
                border-top: 1px solid rgba(0, 0, 0, 0.12);
              }
              .combined-map-legend .legend-title {
                font-weight: 650;
                margin-bottom: 5px;
              }
              .combined-map-legend .year-row {
                display: flex;
                align-items: center;
                gap: 8px;
                margin-bottom: 6px;
              }
              .combined-map-legend input[type="range"] {
                flex: 1;
              }
              .combined-map-legend .year-label {
                min-width: 42px;
                text-align: right;
                font-weight: 650;
              }
              .combined-map-legend .legend-gradient {
                height: 10px;
                border: 1px solid rgba(0, 0, 0, 0.2);
              }
              .combined-map-legend .compaction-gradient {
                background: linear-gradient(to right, #2166ac, #f7f7f7, #b2182b);
              }
              .combined-map-legend .opera-gradient {
                background: linear-gradient(to right, #440154, #21918c, #fde725);
              }
              .combined-map-legend .legend-labels {
                display: flex;
                justify-content: space-between;
                gap: 8px;
                margin-top: 3px;
              }
              .combined-map-legend .missing-row {
                display: flex;
                align-items: center;
                gap: 5px;
                margin-top: 5px;
                color: #4b5563;
              }
              .combined-map-legend .missing-swatch {
                width: 10px;
                height: 10px;
                border-radius: 50%;
                background: #6b7280;
                border: 1px solid #263238;
                display: inline-block;
              }
            </style>
            {% endmacro %}
            {% macro script(this, kwargs) %}
            (function() {
                const map = {{ this._parent.get_name() }};
                const years = {{ this.years|tojson }};
                const markerRefs = [
                    {% for item in this.marker_refs %}
                    {
                        marker: {{ item['marker_name'] }},
                        site: {{ item['site']|tojson }},
                        station: {{ item['station']|tojson }},
                        values: {{ item['values']|tojson }}
                    }{% if not loop.last %},{% endif %}
                    {% endfor %}
                ];
                const vmin = {{ this.vmin }};
                const vmax = {{ this.vmax }};
                const legendTitle = {{ this.legend_title|tojson }};
                const legendUnits = {{ this.legend_units|tojson }};
                const compactionColorScale = chroma.scale(['#2166ac', '#f7f7f7', '#b2182b']).domain([vmin, vmax]);

                function formatCompaction(value) {
                    if (!Number.isFinite(value)) {
                        return 'No data';
                    }
                    return value.toFixed(3) + ' ' + legendUnits;
                }

                function colorForValue(value) {
                    if (!Number.isFinite(value)) {
                        return '#6b7280';
                    }
                    return compactionColorScale(Math.max(vmin, Math.min(vmax, value))).hex();
                }

                function tooltipHtml(item, year, record) {
                    if (!record || !Number.isFinite(record.value)) {
                        return `${item.station}<br>No compaction value by Dec 31, ${year}`;
                    }
                    return `${item.station}<br>Compaction by Dec 31, ${year}: ${formatCompaction(record.value)}<br>Measurement date: ${record.date}`;
                }

                function updateMarkers(year) {
                    for (const item of markerRefs) {
                        const marker = item.marker;
                        if (!marker) {
                            continue;
                        }
                        const record = item.values[String(year)];
                        const value = record && Number.isFinite(record.value) ? Number(record.value) : NaN;
                        marker.setStyle({
                            fillColor: colorForValue(value),
                            color: '#263238',
                            fillOpacity: 0.9,
                            weight: 1
                        });
                        if (marker.getTooltip()) {
                            marker.setTooltipContent(tooltipHtml(item, year, record));
                        } else {
                            marker.bindTooltip(tooltipHtml(item, year, record));
                        }
                    }
                }

                const control = L.control({ position: 'bottomleft' });
                control.onAdd = function() {
                    const div = L.DomUtil.create('div', 'combined-map-legend');
                    const maxIndex = Math.max(0, years.length - 1);
                    div.innerHTML = `
                      <div class="legend-heading">Map legend</div>
                      <div class="legend-section compaction-section">
                        <div class="legend-title">${legendTitle} (${legendUnits})</div>
                        <div class="year-row">
                          <input class="year-slider" type="range" min="0" max="${maxIndex}" step="1" value="${maxIndex}" ${years.length <= 1 ? 'disabled' : ''}>
                          <span class="year-label">${years[maxIndex] || ''}</span>
                        </div>
                        <div class="legend-gradient compaction-gradient"></div>
                        <div class="legend-labels">
                          <span>${formatCompaction(vmin)}</span>
                          <span>${formatCompaction(vmax)}</span>
                        </div>
                        <div class="missing-row"><span class="missing-swatch"></span><span>No value by selected year</span></div>
                      </div>
                      <div class="legend-section opera-section">
                        <div class="legend-title opera-title">OPERA subsidence estimate (mm/year)</div>
                        <div class="legend-gradient opera-gradient"></div>
                        <div class="legend-labels">
                          <span class="opera-min">Loading</span>
                          <span class="opera-max">Loading</span>
                        </div>
                      </div>`;
                    L.DomEvent.disableClickPropagation(div);
                    L.DomEvent.disableScrollPropagation(div);
                    const slider = div.querySelector('.year-slider');
                    const label = div.querySelector('.year-label');
                    slider.addEventListener('input', function() {
                        const year = years[Number(slider.value)];
                        label.textContent = year;
                        updateMarkers(year);
                    });
                    setTimeout(function() {
                        updateMarkers(years[maxIndex]);
                    }, 0);
                    return div;
                };
                control.addTo(map);
            })();
            {% endmacro %}
        """)


### Assemble and Save the Folium Map
With the reusable popup URLs and the COG overlay defined, this cell builds the actual map: base tiles, OPERA raster layer with a raster legend, extensometer markers controlled by a year slider, optional county boundaries, and the saved HTML output.


In [ ]:
MAP_SIMPLIFY_TOLERANCE = float(os.environ.get('MAP_SIMPLIFY_TOLERANCE', '0.005'))
INCLUDE_COUNTY_BOUNDARIES = os.environ.get('INCLUDE_COUNTY_BOUNDARIES', '1') == '1'
OPERA_COG_DATASET_ID = 'houston-opera-subsidence-estimates'
OPERA_COG_RESOURCE_NAME = 'Opera_disp.tif'

opera_cog_url = tutorial_utils.ckan_resource_url(CKAN_URL, OPERA_COG_DATASET_ID, OPERA_COG_RESOURCE_NAME)

years = list(range(int(df['DATE'].dt.year.min()), int(df['DATE'].dt.year.max()) + 1))
year_end_records = []
for site, site_measurements in df.sort_values('DATE').groupby('site'):
    for year in years:
        cutoff = pd.Timestamp(year=year, month=12, day=31)
        through_year = site_measurements[site_measurements['DATE'] <= cutoff]
        if through_year.empty:
            value = None
            date = None
        else:
            latest_record = through_year.iloc[-1]
            value = float(latest_record['CUMULATIVE_COMPACTION'])
            date = latest_record['DATE'].date().isoformat()
        year_end_records.append({
            'name_condensed': site,
            'year': year,
            'value': value,
            'date': date,
        })

year_end_compaction = pd.DataFrame(year_end_records)
compaction_values = year_end_compaction['value'].dropna() if not year_end_compaction.empty else pd.Series(dtype=float)
if compaction_values.empty:
    compaction_min, compaction_max = 0.0, 1.0
else:
    compaction_min = float(compaction_values.min())
    compaction_max = float(compaction_values.max())
    if compaction_min == compaction_max:
        compaction_min -= 0.1
        compaction_max += 0.1

year_value_lookup = {}
for site, site_records in year_end_compaction.groupby('name_condensed'):
    year_value_lookup[site] = {
        str(int(row['year'])): {
            'value': None if pd.isna(row['value']) else float(row['value']),
            'date': None if pd.isna(row['date']) else row['date'],
        }
        for _, row in site_records.iterrows()
    }

initial_year = years[-1]
initial_values = year_end_compaction[year_end_compaction['year'] == initial_year][['name_condensed', 'value', 'date']].rename(
    columns={'value': 'initial_compaction_ft', 'date': 'initial_measurement_date'}
)
sites_for_map = sites_df.merge(initial_values, on='name_condensed', how='left')

m = folium.Map((29.7001, -95.3701), zoom_start=9, tiles="OpenStreetMap", name='Open Street Map')
folium.TileLayer("cartodb positron", name='Carto DB', show=False).add_to(m)

fg_opera = folium.FeatureGroup(name='OPERA subsidence estimates', show=False)
fg_opera.add_to(m)
GeoTiffOverlay(opera_cog_url, fg_opera, opacity=0.65, legend_title='OPERA subsidence estimate', legend_units='mm/year').add_to(m)

fg_extensometers = folium.FeatureGroup(name='extensometers colored by year-end compaction')
marker_refs = []

# Add compaction sites to map. The time slider updates marker colors to show
# cumulative compaction as of December 31 for the selected year.
for _, site_row in sites_for_map.iterrows():
    iframe_src = site_row['popup_url'].replace('&', '&amp;')
    html = f'<iframe src="{iframe_src}" width="540" height="380" frameborder="0" loading="lazy"></iframe>'
    popup = folium.Popup(folium.Html(html, script=True), max_width=570)
    tooltip = site_row['STATION_NM']
    marker = folium.CircleMarker(
        location=[site_row['DEC_LAT_VA'], site_row['DEC_LONG_VA']],
        radius=7,
        color='#263238',
        weight=1,
        fill=True,
        fill_color='#6b7280',
        fill_opacity=0.9,
        tooltip=tooltip,
        popup=popup,
    )
    marker.add_to(fg_extensometers)
    marker_refs.append({
        'marker_name': marker.get_name(),
        'site': site_row['name_condensed'],
        'station': site_row['STATION_NM'],
        'values': year_value_lookup.get(site_row['name_condensed'], {}),
    })
fg_extensometers.add_to(m)
YearEndCompactionControl(
    years,
    marker_refs,
    vmin=compaction_min,
    vmax=compaction_max,
    legend_title='Cumulative compaction by Dec 31',
    legend_units='ft',
).add_to(m)

# Harris County Boundary
county_list = ['Harris', 'Fort Bend', 'Galveston']
if INCLUDE_COUNTY_BOUNDARIES:
    for county in county_list:
        county_geom = tx_gdf_county[tx_gdf_county['CNTY_NM'] == county].copy()
        county_geom.geometry = county_geom.geometry.simplify(MAP_SIMPLIFY_TOLERANCE, preserve_topology=True)
        feature_name = county + ' County'
        if county == 'Harris':
            show_choice = True
        else:
            show_choice = False
        folium.GeoJson(
            county_geom,
            name=feature_name,
            show=show_choice,
            style_function=lambda feature: {
                'fillOpacity': 0.1,
            },
        ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

map_output_path = output_dir / 'index.html'
m.save(map_output_path.as_posix())
map_size_mb = round(tutorial_utils.file_size_mb(map_output_path), 3)
map_output_path, map_size_mb


### Upload the Final Folium Map and Create a CKAN Web View
Upload the saved `index.html` map into the new CKAN dataset created above, then create or update a `webpage_view` so the resource page can render the map inside CKAN via iframe.


In [49]:
map_size_mb = tutorial_utils.file_size_mb(map_output_path)
if map_size_mb > MAX_UPLOAD_MB_WARNING:
    raise ValueError(
        f"index.html is {map_size_mb:.2f} MB. Increase MAP_SIMPLIFY_TOLERANCE or set INCLUDE_COUNTY_BOUNDARIES=0 before uploading."
    )

map_resource = tutorial_utils.upsert_resource_by_name(
    ckan_client,
    html_dataset,
    map_output_path,
    name='index.html',
    description='Folium overview map for the Houston-area extensometer compaction tutorial.',
    format_name='HTML',
    max_upload_mb_warning=MAX_UPLOAD_MB_WARNING,
)
map_resource_url = tutorial_utils.resource_download_url(CKAN_URL, html_dataset, map_resource)
map_view = tutorial_utils.upsert_webpage_view(
    ckan_client,
    map_resource['id'],
    title='Interactive Folium Map',
    description='Embedded CKAN web view for the uploaded Folium map resource.',
)
{
    'map_resource_url': map_resource_url,
    'resource_page_url': f"{CKAN_URL}/dataset/{html_dataset['name']}/resource/{map_resource['id']}",
    'resource_view_id': map_view['id'],
}


{'map_resource_url': 'https://ckan.tacc.utexas.edu/dataset/houston-area-extensometer-compaction-campaign-folium-map/resource/59ae977e-250f-4f1a-8fac-41184997b722/download/index.html',
 'resource_page_url': 'https://ckan.tacc.utexas.edu/dataset/houston-area-extensometer-compaction-campaign-folium-map/resource/59ae977e-250f-4f1a-8fac-41184997b722',
 'resource_view_id': 'cbf46aa3-4c30-4ce9-8e5e-8b3817ebfe13'}